<img src='http://hilpisch.com/tpq_logo.png' width="350px" align="left">
<img src='http://hilpisch.com/taim_logo.png' width="300px" align="right">

# Oanda Master Class

## Strategy Lifecycle

Dr Yves J Hilpisch | The Python Quants | The AI Machine

<td><a href="http://tpq.io" target="_blank">http://tpq.io</a> | <a href="http://aimachine.io" target="_blank">http://aimachine.io</a> | <a href="http://twitter.com/dyjh" target="_blank">@dyjh</a> | <a href="mailto:team@tpq.io">team@tpq.io</a></td>

## Imports

In [ ]:
!git clone https://github.com/tpq-classes/python_for_algo_trading_addon.git
import sys
sys.path.append('python_for_algo_trading_addon')


In [ ]:
!pip install git+https://github.com/yhilpisch/tpqoa

In [ ]:
import numpy as np
import pandas as pd
from pylab import plt
plt.style.use('seaborn-v0_8')

## API Connection

In [ ]:
import tpqoa

In [ ]:
api = tpqoa.tpqoa('/content/python_for_algo_trading_addon/pyalgo.cfg')

## The Data

In [ ]:
instrument = 'EUR_USD'
start = '2020-03-17'
end = '2020-03-19'
granularity = 'S5'
price = 'M'

In [ ]:
%%time
data = api.get_history(instrument, start, end,
                       granularity, price, localize=True)

In [ ]:
data.info()

In [ ]:
data['c'].plot(figsize=(10, 6));

## Simple Moving Averages

In [ ]:
data['r'] = np.log(data['c'] / data['c'].shift(1))

In [ ]:
data['SMA1'] = data['c'].rolling(3).mean().shift(1)

In [ ]:
data['SMA2'] = data['c'].rolling(6).mean().shift(1)

In [ ]:
data.dropna(inplace=True)

In [ ]:
data[['c', 'SMA1', 'SMA2']].iloc[-150:].plot(figsize=(10, 6));

In [ ]:
data['p'] = np.where(data['SMA1'] > data['SMA2'], 1, -1)

In [ ]:
sum(data['p'].diff() != 0)

In [ ]:
data['s'] = data['p'] * data['r']

In [ ]:
data.head()

In [ ]:
data[['r', 's']].sum().apply(np.exp)  # gross performance

In [ ]:
data[['r', 's']].cumsum().apply(np.exp).plot(figsize=(10, 6));  # gross performance over time

## Streaming Data

In [ ]:
api.stream_data('EUR_USD', stop=10)

In [ ]:
api.on_success??

## Deployment

In [ ]:
#api.stream_data??

In [ ]:
class SMAAlgoTrader(tpqoa.tpqoa):
    def __init__(self, config_file, granularity, SMA1, SMA2, units):
        super(SMAAlgoTrader, self).__init__(config_file)
        self.granularity = granularity
        self.SMA1 = SMA1
        self.SMA2 = SMA2
        self.units = units
        self.min_length = self.SMA2
        self.position = 0
        self.pls = list()
        self.tick_data = pd.DataFrame()
    def _resample_data(self):
        self.data = self.tick_data.resample(self.granularity,
                                label='right').last().ffill()
    def _prepare_data(self):
        self.data['SMA1'] = self.data['mid'].rolling(self.SMA1).mean()
        self.data['SMA2'] = self.data['mid'].rolling(self.SMA2).mean()
        self.data['p'] = np.where(self.data['SMA1'] > self.data['SMA2'],
                                  'long', 'short')
    def report_trade(self, order, side):
        print(order)
        time = order['time']
        units = order['units']
        price = order['price']
        pl = float(order['pl'])
        self.pls.append(pl)
        cpls = sum(self.pls)
        print(2 * '\n' + 90 * '=')
        print(f'{time} | *** GOING {side} ***')
        print(f'{time} | units={units} | price={price}| P&L={pl:.4f} | CP&L={cpls:.4f}')
        print(90 * '=' + '\n')
    def on_success(self, time, bid, ask):
        print(self.ticks, end=' ')
        df = pd.DataFrame({'ask': ask, 'bid': bid,
                           'mid': (ask + bid) / 2},
                         index=[pd.Timestamp(time)])
        self.tick_data = pd.concat((self.tick_data, df))
        self._resample_data()
        self._prepare_data()
        if len(self.data.iloc[:-1]) > self.min_length:
            self.min_length += 1
            if self.position in [0, -1]:
                if self.data['p'].iloc[-2] == 'long':
                    order = self.create_order(self.stream_instrument,
                                units=(1 - self.position) * self.units,
                                             suppress=True, ret=True)
                    self.report_trade(order, 'LONG')
                    self.position = 1
            elif self.position in [0, 1]:
                if self.data['p'].iloc[-2] == 'short':
                    order = self.create_order(self.stream_instrument,
                                units=-(1 + self.position) * self.units,
                                             suppress=True, ret=True)
                    self.report_trade(order, 'SHORT')
                    self.position = -1

In [ ]:
sma = SMAAlgoTrader('/content/python_for_algo_trading_addon/pyalgo.cfg', granularity='5s', SMA1=3, SMA2=6, units=1000)

In [ ]:
sma.get_instruments()[:5]

In [ ]:
sma.stream_data('EUR_USD', stop=500)  # streaming & trading
order = sma.create_order(sma.stream_instrument,
                         units=-sma.position * sma.units,
                         suppress=True, ret=True)  # closing the final position
sma.report_trade(order, 'NEUTRAL')  # reporting the final trade

<img src='http://hilpisch.com/tpq_logo.png' width="350px" align="left">
<img src='http://hilpisch.com/taim_logo.png' width="300px" align="right">